# Phase 5: Model Training & CV Tuning

This notebook loads the processed features parquet, performs a strict well-based split to avoid data leakage, normalizes inputs for distance-based classifiers (StandardScaler for KNN), trains and tunes five classifiers (KNN, Decision Tree, Random Forest, XGBoost, and LightGBM) on the 12-feature and 19-feature datasets, and saves the trained models.

In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('../'))
from src.features import scale_features
from src.models import train_and_save_all

# 1. Load engineered features
df = pd.read_parquet('../data/interim/processed_features.parquet')
print(f'Features DataFrame shape: {df.shape}')

## 1. Train / Test Split by Well

To prevent data leakage from high spatial correlations in a single well, we split by `WELL_ID` rather than randomly shuffling rows. 8 wells go to train, and 2 wells are held out for testing.

In [ ]:
wells = df['WELL_ID'].unique()
train_wells = wells[:8]
test_wells = wells[8:]

df_train = df[df['WELL_ID'].isin(train_wells)].copy()
df_test = df[df['WELL_ID'].isin(test_wells)].copy()

print(f'Train Wells: {train_wells}')
print(f'Test Wells: {test_wells}')
print(f'Train samples: {len(df_train)}, Test samples: {len(df_test)}')

## 2. Feature Sets Definitions

Define original (12 features) vs wavelet-transformed (19 features) datasets.

In [ ]:
original_cols = ['DEPTH_MD', 'CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'BS']
wavelet_cols = original_cols + [f'{col}_CWT' for col in ['GR', 'NPHI', 'SP', 'RDEP', 'RHOB', 'DTC', 'PEF']]

print(f'Original features count: {len(original_cols)}')
print(f'Wavelet features count: {len(wavelet_cols)}')

## 3. Standard Scaling

Distance-based classifiers like KNN require scaling. Tree models do not. We fit standard scalers strictly on train, transforming test.

In [ ]:
X_train_orig_scaled, X_test_orig_scaled, scaler_orig = scale_features(df_train, df_test, original_cols)
X_train_wav_scaled, X_test_wav_scaled, scaler_wav = scale_features(df_train, df_test, wavelet_cols)
print('Scalers fitted and transformed successfully.')

## 4. Train Models on Original 12-Feature Dataset

Trains and saves KNN, DT, RF, XGB, and LGB. Tuning uses 5-fold GroupKFold CV.

In [ ]:
y_train = df_train['LITHOLOGY'].values
groups = df_train['WELL_ID'].values

models_original = train_and_save_all(
    X_train_scaled=X_train_orig_scaled,
    X_train_raw=df_train[original_cols],
    y_train=y_train,
    groups=groups,
    feature_set_name='original'
)

## 5. Train Models on Wavelet 19-Feature Dataset

Trains and saves the five classifiers on the 19 features including PyWavelets CWT.

In [ ]:
models_wavelet = train_and_save_all(
    X_train_scaled=X_train_wav_scaled,
    X_train_raw=df_train[wavelet_cols],
    y_train=y_train,
    groups=groups,
    feature_set_name='wavelet'
)